# BıRAG — sohbet modeli: `v0.1.1` ile eğit, dene, GGUF kaydet

Unsloth'un [Gemma4 (E4B) Text](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Gemma4_(E4B)-Text.ipynb) akışı: **kur → yükle → veri → LoRA → eğit → sohbet dene → GGUF**.
Çıktı: sohbet uygulamasında (llama.cpp / Ollama) kullanılacak **tek bir model**.

*Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4 ya da A100***, sonra *Tümünü çalıştır*. Oturum koparsa yeniden *Tümünü çalıştır* — eğitim son kontrol noktasından sürer, bitmişse atlanır.

⚠️ **Bu defter deney değildir.** `v0.1.0` ↔ `v0.1.1` kıyası `notebooks/v011-basit-egitim.ipynb`'de (ön kayıt EK-3). Ana hat hâlâ `v0.1.0`; `v0.1.1` deneyin hükmünden **önce** seçildi (kullanıcı kararı, 2026-09-30).

⛔ **Kriz davranışı:** eğitim verisinde kriz dilimi yok (uzman onayı bekliyor). Bu model, kriz protokolü ve güvenlik katmanı olmadan gerçek kullanıcıya açılmamalı.

### 1. Drive ve kurulum

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q unsloth

### 2. Parametreler
Tarif `e3`'ün tarifi (deneyle aynı): bf16 · üstten 8 dil katmanının `q_proj`/`o_proj`'u · r 8 / alpha 160 · LR 1e-5 sabit · 2538 adım.

In [ ]:
VERI     = "datasets/v0.1.1/train.jsonl"
TOHUM    = 7
MODEL_ID = "unsloth/gemma-4-E4B-it"
DRIVE    = "/content/drive/MyDrive/birag-sohbet"
REPO     = "/content/birag-data-generation"
GGUF_NICEMLEME = "Q8_0"        # Unsloth belgesine göre Gemma 4 için şimdilik: "Q8_0" · "BF16" · "F16"

R, LORA_ALPHA, UST_KATMAN = 8, 160, 8
ADIM, LR, MAX_SEQ = 2538, 1e-5, 2048

### 3. Depo

In [ ]:
import os, sys, json
if not os.path.exists(REPO):
    !git clone -q https://github.com/enveryildirim/birag-data-generation.git {REPO}
sys.path.insert(0, f"{REPO}/src")

from pathlib import Path
KOSU = Path(DRIVE) / f"v011-t{TOHUM}"
ADAPTER = KOSU / "adapter"
KOSU.mkdir(parents=True, exist_ok=True)

### 4. Modeli yükle ve şablonlar
İki şablon var:
- **eğitim şablonu** `configs/chat_template_train.jinja` (K44) — Gemma'nın resmi şablonundan tek farkı son model turunun düşünmesini silmemesi (eğitim hedefi o).
- **resmi şablon** — modelle gelen. Sohbette son tur hep kullanıcıda olduğu için ikisi **aynı istemi** üretir (7. hücre bunu denetler). GGUF'a resmi şablon yazılır.

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(model_name=MODEL_ID, max_seq_length=MAX_SEQ, dtype=torch.bfloat16,
                                             load_in_4bit=False, full_finetuning=False)
tok = getattr(tokenizer, "tokenizer", tokenizer)            # Gemma 4 işlemci döndürür → metin tokenizer'ı
RESMI_SABLON = tok.chat_template
EGITIM_SABLONU = Path(REPO, "configs/chat_template_train.jinja").read_text(encoding="utf-8")

import colab_egitim as ce
print("şablon denetimi:", ce.sablon_referansi_denetle(tokenizer), "dizge referansla aynı")   # eğitim şablonunu kurar
print("GPU:", torch.cuda.get_device_name(0))

### 5. Veri
Bölme `src/train.py` ile (`random.Random(7)`, %20 doğrulama ⇒ 846 / 212). Kayıp yalnız **son** asistan turunda.

In [ ]:
import tempfile
from train import prepare_data

with tempfile.TemporaryDirectory() as d:
    n_tr, n_va = prepare_data(Path(REPO, VERI), Path(d), 0.2, 7)
    oku = lambda ad: [json.loads(x)["messages"] for x in open(f"{d}/{ad}.jsonl") if x.strip()]
    egitim_msj, dogrulama_msj = oku("train"), oku("valid")
print(f"{n_tr} eğitim / {n_va} doğrulama")

def kodla(msgs):
    tam = tok(tok.apply_chat_template(msgs, tokenize=False), add_special_tokens=False)["input_ids"]
    on  = tok(tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True),
              add_special_tokens=False)["input_ids"]
    assert tam[:len(on)] == on and len(tam) <= MAX_SEQ
    return {"input_ids": tam, "attention_mask": [1] * len(tam), "labels": [-100] * len(on) + tam[len(on):]}

### 6. Eğit

In [ ]:
import re, time, transformers
from datasets import Dataset

if (ADAPTER / "adapter_config.json").exists():
    print("✅ zaten eğitilmiş — 7. hücre adapter'ı yükleyecek")
else:
    transformers.set_seed(TOHUM)
    katman = {}
    for ad, _ in model.named_modules():
        m = re.search(r"\.layers\.(\d+)\.self_attn\.(q_proj|o_proj)$", ad)
        if m and not re.search(r"vision|audio", ad):
            katman.setdefault(int(m.group(1)), []).append(ad)
    HEDEF = [ad for k in sorted(katman)[-UST_KATMAN:] for ad in katman[k]]

    model = FastModel.get_peft_model(model, r=R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias="none",
                                     target_modules=HEDEF, use_gradient_checkpointing="unsloth", random_state=TOHUM)
    lora = [ad for ad, mod in model.named_modules() if hasattr(mod, "lora_A") and len(mod.lora_A)]
    assert len(lora) == len(HEDEF) and not any(re.search("vision|audio", a) for a in lora), f"LoRA yanlış yerde: {lora[:3]}"
    model.print_trainable_parameters()

    trainer = transformers.Trainer(
        model=model,
        train_dataset=Dataset.from_list([kodla(m) for m in egitim_msj]),
        eval_dataset=Dataset.from_list([kodla(m) for m in dogrulama_msj]),
        data_collator=transformers.DataCollatorForSeq2Seq(tok, padding=True),
        args=transformers.TrainingArguments(
            output_dir=str(KOSU / "ckpt"), per_device_train_batch_size=1, per_device_eval_batch_size=1,
            max_steps=ADIM, learning_rate=LR, lr_scheduler_type="constant", warmup_steps=0,
            optim="adamw_torch", weight_decay=0.01, max_grad_norm=0.0, bf16=True,
            logging_steps=50, eval_strategy="steps", eval_steps=423, save_strategy="steps", save_steps=846,
            seed=TOHUM, report_to="none", remove_unused_columns=False))
    t0 = time.time()
    trainer.train(resume_from_checkpoint=any((KOSU / "ckpt").glob("checkpoint-*")))
    print(f"✅ eğitim {round((time.time() - t0) / 60, 1)} dk")

    tok.chat_template = RESMI_SABLON                     # kaydedilen tokenizer resmi şablonu taşısın
    if hasattr(tokenizer, "chat_template"): tokenizer.chat_template = RESMI_SABLON
    model.save_pretrained(str(ADAPTER)); tokenizer.save_pretrained(str(ADAPTER))
    del trainer

### 7. Sohbet dene
Deneme konuşması `evals/cok_turlu.jsonl`'in ilki (4 kullanıcı turu, eğitimdeki sistem istemiyle). Gemma'nın önerdiği ayar: `temperature 1.0 · top_p 0.95 · top_k 64`. Düşünme kapalı; açmak için sistem isteminin başına `<|think|>` eklenir.
Geçmişe yalnız **görünen cevap** konur, düşünme konmaz (Gemma 4 kuralı — uygulamada da böyle olmalı).

In [ ]:
if not hasattr(model, "peft_config"):                 # eğitim atlandıysa taban + adapter'ı yükle
    model, tokenizer = FastModel.from_pretrained(model_name=str(ADAPTER), max_seq_length=MAX_SEQ,
                                                 dtype=torch.bfloat16, load_in_4bit=False)
    tok = getattr(tokenizer, "tokenizer", tokenizer)
tok.chat_template = RESMI_SABLON
if hasattr(tokenizer, "chat_template"): tokenizer.chat_template = RESMI_SABLON
FastModel.for_inference(model)

konusma = json.loads(open(f"{REPO}/evals/cok_turlu.jsonl", encoding="utf-8").readline())
SISTEM = konusma["messages"][0]["content"]

# denetim: çok turlu bir sohbet isteminde eğitim şablonu ile resmi şablon AYNI metni üretiyor mu
ornek = [{"role": "system", "content": SISTEM}, {"role": "user", "content": "a"},
         {"role": "assistant", "content": "b"}, {"role": "user", "content": "c"}]
resmi = tok.apply_chat_template(ornek, tokenize=False, add_generation_prompt=True)
tok.chat_template = EGITIM_SABLONU
egitim = tok.apply_chat_template(ornek, tokenize=False, add_generation_prompt=True)
tok.chat_template = RESMI_SABLON
assert resmi == egitim, "⛔ iki şablon sohbet isteminde farklı — GGUF'a eğitim şablonu yazılmalı, durun"
print("✅ sohbet isteminde eğitim şablonu = resmi şablon")

from golden_eval import cikti_ayir
def cevapla(mesajlar, max_new_tokens=512):
    x = tok.apply_chat_template(mesajlar, add_generation_prompt=True, tokenize=True,
                                return_dict=True, return_tensors="pt").to("cuda")
    y = model.generate(**x, max_new_tokens=max_new_tokens, temperature=1.0, top_p=0.95, top_k=64)
    ham = tok.decode(y[0, x["input_ids"].shape[1]:], skip_special_tokens=False)
    ham = ham.split("<turn|>")[0]
    return cikti_ayir(ham)[1]

mesajlar = [{"role": "system", "content": SISTEM}]
for u in konusma["kullanici_turlari"]:
    mesajlar.append({"role": "user", "content": u})
    c = cevapla(mesajlar)
    mesajlar.append({"role": "assistant", "content": c})
    print(f"👤 {u}\n🤖 {c}\n")

### 8. GGUF kaydet
LoRA tabana birleştirilir ve `GGUF_NICEMLEME` ile yazılır (Unsloth llama.cpp'yi kendisi kurar — birkaç dakika sürer). Önce yerel diske, sonra Drive'a kopyalanır. Yanına **sistem istemi** ve bir Ollama `Modelfile` konur.

In [ ]:
import shutil, glob
YEREL = "/content/birag-gguf"
model.save_pretrained_gguf(YEREL, tokenizer, quantization_method=GGUF_NICEMLEME)

HEDEF_DIZIN = Path(DRIVE) / "gguf"; HEDEF_DIZIN.mkdir(parents=True, exist_ok=True)
ggufs = sorted(glob.glob(f"{YEREL}*/*.gguf") + glob.glob(f"{YEREL}*.gguf"))
assert ggufs, f"⛔ GGUF bulunamadı: {os.listdir('/content')}"
for f in ggufs:
    shutil.copy(f, HEDEF_DIZIN); print("✅", HEDEF_DIZIN / Path(f).name, f"{os.path.getsize(f)/1e9:.1f} GB")

(HEDEF_DIZIN / "sistem_istemi.txt").write_text(SISTEM, encoding="utf-8")
ana = next(Path(f).name for f in ggufs if "mmproj" not in Path(f).name)
(HEDEF_DIZIN / "Modelfile").write_text(
    f"FROM ./{ana}\n"
    f'SYSTEM """{SISTEM}"""\n'
    "PARAMETER temperature 1.0\nPARAMETER top_p 0.95\nPARAMETER top_k 64\n", encoding="utf-8")
print("✅ sistem_istemi.txt · Modelfile")

### 9. Uygulamada kullanım

Drive'daki `birag-sohbet/gguf/` dizinini indirin.

**llama.cpp** (GGUF'taki şablonu kullanır — `--jinja` şart):
```bash
./llama-server -m <model>.gguf --jinja --temp 1.0 --top-p 0.95 --top-k 64 --port 8001
```
Sistem istemi olarak `sistem_istemi.txt`'in içeriğini gönderin.

**Ollama:**
```bash
ollama create birag -f Modelfile
ollama run birag
```
⚠️ Ollama kendi şablonunu kullanabilir; cevaplar tuhafsa llama.cpp `--jinja` ile karşılaştırın (Unsloth: dışa aktarılan model başka bir çalışma ortamında kötü davranıyorsa en sık neden yanlış şablon / EOS).

**Uygulamaya kurallar:** sistem istemi eğitimdekiyle aynı · geçmişe yalnız görünen cevap (düşünme blokları değil) · düşünme kapalı (açmak için sistem isteminin başına `<|think|>`; açıkken düşünme uzun olabilir, K46).